# Notebook 03: Responsible AI & GenAI Integration
## Credit Risk Scoring untuk UMKM Indonesia

**Tujuan Notebook ini:**
1. **SHAP Analysis** — Explainability: mengapa model memutuskan approve/reject?
2. **Fairness Analysis** — Apakah model adil antar kelompok pendapatan?
3. **GitHub Models (GPT-4o-mini)** — GenAI untuk narasi, rekomendasi, dan policy advisor

> 🏦 **Framing:** Model ini adalah tools inklusi keuangan — bukan hanya prediksi risiko,
> tapi juga memberikan *penjelasan* dan *jalan keluar* bagi UMKM yang ditolak.

---
**Azure Setup:** GitHub Models endpoint (compatible dengan Azure OpenAI untuk production)
```
Endpoint : https://models.inference.ai.azure.com
Model Dev: gpt-4o-mini
Model Demo: gpt-4o
Auth      : GitHub Personal Access Token (PAT)
```

## Section 0: Setup & Load Artifacts

In [ ]:
# Install dependencies (jalankan sekali)
import subprocess, sys

packages = ["shap", "openai", "matplotlib", "seaborn", "scikit-learn",
            "pandas", "numpy", "azureml-sdk", "mlflow"]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ Semua package terinstall")

In [ ]:
# Core Imports
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import mlflow
from openai import OpenAI

warnings.filterwarnings('ignore')
shap.initjs()

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False
})

print("✅ Import sukses")

In [ ]:
# Load Processed Dataset
# Sesuaikan path dengan lokasi dataset kamu
DATASET_PATH = "./processed_credit_risk.csv"  # ganti jika perlu

# Opsi: Load dari Azure ML Data Asset
# from azureml.core import Workspace, Dataset
# ws = Workspace.from_config()
# dataset = Dataset.get_by_name(ws, name='credit_risk_processed')
# df = dataset.to_pandas_dataframe()

df = pd.read_csv(DATASET_PATH)

print(f"✅ Dataset loaded: {df.shape[0]:,} baris x {df.shape[1]} kolom")
print(f"\nTarget distribution:")
print(df['loan_status'].value_counts(normalize=True).mul(100).round(1).to_string())

In [ ]:
# Define Feature Columns (9 fitur dari 4C framework)
FEATURE_COLS = [
    'debt_service_ratio',          # Capacity
    'loan_to_income_ratio',        # Capacity
    'income_log',                  # Capacity
    'loan_percent_income',         # Capital
    'interest_risk_band',          # Conditions
    'loan_grade_encoded',          # Conditions
    'cb_person_default_on_file_encoded',  # Character
    'loan_amnt_log',               # Additional
    'person_age'                   # Additional
]

TARGET_COL = 'loan_status'

available_features = [f for f in FEATURE_COLS if f in df.columns]
print(f"✅ Fitur tersedia: {len(available_features)}/{len(FEATURE_COLS)}")
print(f"   {available_features}")

X = df[available_features]
y = df[TARGET_COL]

In [ ]:
# Load Best Model dari Azure ML Registry
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

try:
    from azureml.core import Workspace, Model
    import joblib
    ws = Workspace.from_config()
    model_obj = Model(ws, name='credit-risk-model')  # sesuaikan nama model
    model_path = model_obj.download(target_dir='.', exist_ok=True)
    model = joblib.load('./model.pkl')
    print(f"✅ Model loaded dari Azure ML Registry")
    print(f"   Model type: {type(model).__name__}")

except Exception as e:
    print(f"⚠️ Azure ML load gagal: {e}")
    print("   Fallback: training GradientBoosting sebagai proxy...")
    from sklearn.ensemble import GradientBoostingClassifier
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = GradientBoostingClassifier(n_estimators=200, random_state=42)
    model.fit(X_tr, y_tr)
    print("   ✅ Fallback model trained")

# Train/test split untuk analisis
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f"\n📊 Model Performance Validation:")
print(f"   AUC : {roc_auc_score(y_test, y_prob):.4f}")
print(f"   F1  : {f1_score(y_test, y_pred):.4f}")

---
## Section 1: SHAP Analysis — Explainability

**Konsep:** SHAP (SHapley Additive exPlanations) mengukur kontribusi setiap fitur
terhadap prediksi model secara *fair* — terinspirasi dari game theory.

- **Summary Plot** → gambaran global: fitur mana yang paling penting?
- **Waterfall Plot** → penjelasan lokal: mengapa *individu ini* di-approve/reject?

In [ ]:
# Hitung SHAP Values
print("⏳ Menghitung SHAP values (1-3 menit)...")

# Sample 1000 untuk efisiensi
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), size=min(1000, len(X_test)), replace=False)
X_sample = X_test.iloc[sample_idx].reset_index(drop=True)

try:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    print(f"✅ SHAP values dihitung (TreeExplainer): {sv.shape}")

except Exception as e:
    print(f"⚠️ TreeExplainer gagal: {e}\n   Pakai KernelExplainer...")
    explainer = shap.KernelExplainer(model.predict_proba, shap.sample(X_train, 100))
    shap_values_raw = explainer.shap_values(X_sample)
    sv = shap_values_raw[1]
    print(f"✅ SHAP values dihitung (KernelExplainer)")

In [ ]:
# Plot 1: SHAP Summary Plot (Global Feature Importance)
fig, ax = plt.subplots(figsize=(10, 6))

shap.summary_plot(
    sv, X_sample,
    feature_names=available_features,
    plot_type='dot',
    max_display=9,
    show=False,
    color_bar=True
)

plt.title(
    'SHAP Summary Plot — Feature Impact pada Prediksi Credit Risk\n'
    '(Merah = nilai tinggi | Kanan = meningkatkan risiko default)',
    fontsize=12, pad=15
)
plt.tight_layout()
plt.savefig('shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print("💾 Saved: shap_summary_plot.png")
print("\n📌 Insight:")
print("   debt_service_ratio tinggi → MERAH → kanan → risiko default NAIK")
print("   income_log tinggi → MERAH → kiri → risiko default TURUN")

In [ ]:
# Identify high-risk dan low-risk samples
prob_sample = model.predict_proba(X_sample)[:, 1]
high_risk_idx = np.argmax(prob_sample)
low_risk_idx  = np.argmin(prob_sample)

print(f"📍 HIGH RISK (idx {high_risk_idx}) — P(default) = {prob_sample[high_risk_idx]:.1%}")
print(X_sample.iloc[high_risk_idx].to_string())
print(f"\n📍 LOW RISK (idx {low_risk_idx}) — P(default) = {prob_sample[low_risk_idx]:.1%}")
print(X_sample.iloc[low_risk_idx].to_string())

In [ ]:
# Helper: Waterfall / bar plot yang robust
def plot_waterfall(shap_vals, sample_data, title, filename, expected_val=None):
    """Plot SHAP waterfall, fallback ke bar plot jika perlu."""
    try:
        base = (expected_val[1] if isinstance(expected_val, (list, np.ndarray))
                else expected_val) if expected_val is not None else 0
        exp = shap.Explanation(
            values=shap_vals,
            base_values=base,
            data=sample_data.values,
            feature_names=available_features
        )
        plt.figure(figsize=(10, 5))
        shap.waterfall_plot(exp, max_display=9, show=False)
    except Exception:
        shap_df = pd.DataFrame({'feature': available_features, 'shap': shap_vals})
        shap_df = shap_df.reindex(shap_df['shap'].abs().sort_values().index)
        colors = ['#d73027' if v > 0 else '#4575b4' for v in shap_df['shap']]
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.barh(shap_df['feature'], shap_df['shap'], color=colors)
        ax.axvline(x=0, color='black', lw=0.8)
        ax.set_xlabel('SHAP Value')
        red_p = mpatches.Patch(color='#d73027', label='Meningkatkan risiko')
        blue_p = mpatches.Patch(color='#4575b4', label='Menurunkan risiko')
        ax.legend(handles=[red_p, blue_p])

    plt.title(title, fontsize=12, pad=15)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"💾 Saved: {filename}")

# Plot Waterfall — HIGH RISK
plot_waterfall(
    sv[high_risk_idx],
    X_sample.iloc[high_risk_idx],
    f'Waterfall Plot — Peminjam DITOLAK (P(default) = {prob_sample[high_risk_idx]:.1%})',
    'shap_waterfall_high_risk.png',
    explainer.expected_value
)

# Plot Waterfall — LOW RISK
plot_waterfall(
    sv[low_risk_idx],
    X_sample.iloc[low_risk_idx],
    f'Waterfall Plot — Peminjam DISETUJUI (P(default) = {prob_sample[low_risk_idx]:.1%})',
    'shap_waterfall_low_risk.png',
    explainer.expected_value
)

# Global feature importance
shap_importance = pd.DataFrame({
    'feature': available_features,
    'mean_abs_shap': np.abs(sv).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print("\n📊 Global Feature Importance (Mean |SHAP|):")
print(shap_importance.to_string(index=False))

---
## Section 2: Fairness Analysis — Bias Detection

**Pertanyaan kunci:** Apakah model ini *adil* untuk semua kelompok UMKM?
- Apakah pengusaha berpendapatan rendah (Q1) diperlakukan lebih ketat?
- Apakah umur mempengaruhi keputusan kredit secara tidak proporsional?

> Ini langsung menjawab tema **Inklusi Keuangan** dari datathon.

In [ ]:
# Fairness Setup
df_fair = X_test.copy()
df_fair['true_label']  = y_test.values
df_fair['pred_label']  = y_pred
df_fair['pred_proba']  = y_prob
df_fair['correct']     = (df_fair['true_label'] == df_fair['pred_label']).astype(int)

# Income Quartile Groups
df_fair['income_quartile'] = pd.qcut(
    df_fair['income_log'],
    q=4,
    labels=['Q1 (Rendah)', 'Q2 (Bawah-Menengah)', 'Q3 (Atas-Menengah)', 'Q4 (Tinggi)']
)

# Age Groups
df_fair['age_group'] = pd.cut(
    df_fair['person_age'],
    bins=[0, 25, 35, 45, 100],
    labels=['<=25 thn', '26-35 thn', '36-45 thn', '>45 thn']
)

print("✅ Fairness groups created")
print(df_fair['income_quartile'].value_counts().sort_index())

In [ ]:
# Fairness Metrics Function
def compute_fairness_metrics(df_group, group_col):
    results = []
    for group in df_group[group_col].cat.categories:
        sub = df_group[df_group[group_col] == group]
        if len(sub) < 10:
            continue
        tp = ((sub['pred_label']==1) & (sub['true_label']==1)).sum()
        tn = ((sub['pred_label']==0) & (sub['true_label']==0)).sum()
        fp = ((sub['pred_label']==1) & (sub['true_label']==0)).sum()
        fn = ((sub['pred_label']==0) & (sub['true_label']==1)).sum()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
        results.append({
            'Kelompok': str(group),
            'N': len(sub),
            'Actual Default Rate': f"{sub['true_label'].mean():.1%}",
            'Avg Risk Score': f"{sub['pred_proba'].mean():.3f}",
            'False Positive Rate': f"{fpr:.3f}",
            'False Negative Rate': f"{fnr:.3f}",
            'Accuracy': f"{(tp+tn)/len(sub):.3f}",
            '_actual_dr': sub['true_label'].mean(),
            '_avg_score': sub['pred_proba'].mean()
        })
    return pd.DataFrame(results)

fairness_income = compute_fairness_metrics(df_fair, 'income_quartile')
fairness_age    = compute_fairness_metrics(df_fair, 'age_group')

display_cols = ['Kelompok', 'N', 'Actual Default Rate', 'Avg Risk Score',
                'False Positive Rate', 'False Negative Rate', 'Accuracy']

print("📊 Fairness Metrics — Income Quartile:")
print(fairness_income[display_cols].to_string(index=False))

print("\n📊 Fairness Metrics — Age Group:")
print(fairness_age[display_cols].to_string(index=False))

In [ ]:
# Fairness Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel kiri: Default rate per income quartile
income_labels = fairness_income['Kelompok'].tolist()
actual_rates  = fairness_income['_actual_dr'].tolist()
risk_scores   = fairness_income['_avg_score'].tolist()

x = np.arange(len(income_labels))
width = 0.35
axes[0].bar(x - width/2, actual_rates, width, label='Actual Default Rate',
            color='#d73027', alpha=0.8)
bars2 = axes[0].bar(x + width/2, risk_scores, width, label='Avg Predicted Risk Score',
            color='#fc8d59', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(income_labels, fontsize=8, rotation=10)
axes[0].set_ylabel('Rate / Score')
axes[0].set_title('Default Rate & Risk Score\nper Kelompok Pendapatan', fontsize=11)
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Panel kanan: Disparity gap
q1_rate = fairness_income.iloc[0]['_actual_dr']
q4_rate = fairness_income.iloc[-1]['_actual_dr']
gap_income = q1_rate - q4_rate

age_rates = fairness_age['_actual_dr'].tolist()
gap_age = max(age_rates) - min(age_rates)

categories = ['Income Gap\n(Q1 vs Q4)', 'Age Gap\n(Max vs Min)']
gaps = [gap_income, gap_age]
colors_gap = ['#d73027' if g > 0.15 else '#fee090' if g > 0.05 else '#91bfdb' for g in gaps]

bars_gap = axes[1].bar(categories, gaps, color=colors_gap, alpha=0.9, width=0.5)
axes[1].axhline(y=0.05, color='gray', linestyle='--', lw=1, label='Threshold rendah (5%)')
axes[1].axhline(y=0.15, color='red',  linestyle='--', lw=1, label='Threshold tinggi (15%)')
axes[1].set_ylabel('Disparity Gap')
axes[1].set_title('Fairness Disparity Gap\n(lebih kecil = lebih adil)', fontsize=11)
axes[1].legend(fontsize=8)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

for bar, val in zip(bars_gap, gaps):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                 f'{val:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('Fairness Analysis — Credit Risk Model | Inklusi Keuangan UMKM Indonesia',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fairness_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Fairness Summary:")
print(f"   Income Gap (Q1 vs Q4) : {gap_income:.1%} {'⚠️ SIGNIFIKAN' if gap_income > 0.15 else '✅ Wajar'}")
print(f"   Age Gap (Max vs Min)  : {gap_age:.1%} {'⚠️ SIGNIFIKAN' if gap_age > 0.15 else '✅ Wajar (Age-Blind)'}")
print("💾 Saved: fairness_analysis.png")

---
## Section 3: GitHub Models Integration — GPT-4o-mini

**Arsitektur GenAI:**
```
SHAP Results     --> Context Builder --> GPT-4o-mini --> Indonesian Explanation
Borrower Profile --> Prompt Engineer --> GPT-4o-mini --> Counterfactual Advice
Fairness Data    --> Policy Context  --> GPT-4o-mini --> Fairness Explanation
Policy Query     --> Smart Advisor   --> GPT-4o-mini --> Strategic Recommendation
```

> **Untuk Presentasi:** GitHub Models (dev) --> Azure OpenAI (production enterprise)  
> OpenAI-compatible API = zero code change untuk migration!

In [ ]:
# GitHub Models Client Setup
import os
from openai import OpenAI

# Set GitHub PAT sebagai environment variable
# Terminal: export GITHUB_TOKEN='ghp_your_token_here'
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

if not GITHUB_TOKEN:
    print("⚠️ GITHUB_TOKEN tidak ditemukan!")
    print("   Set: import os; os.environ['GITHUB_TOKEN'] = 'ghp_...'")
    GITHUB_TOKEN = input("Masukkan GitHub PAT: ").strip()

# Client — compatible dengan Azure OpenAI SDK
client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=GITHUB_TOKEN
)

ACTIVE_MODEL = "gpt-4o-mini"  # ganti ke "gpt-4o" untuk demo premium

# Test koneksi
try:
    test = client.chat.completions.create(
        model=ACTIVE_MODEL,
        messages=[{"role": "user", "content": "Balas: KONEKSI OK"}],
        max_tokens=20
    )
    print(f"✅ GitHub Models connected | Model: {ACTIVE_MODEL}")
    print(f"   Response: {test.choices[0].message.content}")
except Exception as e:
    print(f"❌ Koneksi gagal: {e}")

In [ ]:
# Helper: Call GitHub Models
def call_genai(system_prompt: str, user_prompt: str, max_tokens: int = 800) -> str:
    """Generic wrapper untuk GitHub Models API call."""
    try:
        response = client.chat.completions.create(
            model=ACTIVE_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}
            ],
            max_tokens=max_tokens,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"[Error: {e}]"

print("✅ Helper function siap")

### 3A: Narasi SHAP Otomatis dalam Bahasa Indonesia

In [ ]:
# 3A: SHAP Narrative Generator
SYSTEM_SHAP = """Kamu adalah AI analis kredit senior untuk UMKM Indonesia.
Tugasmu adalah menjelaskan hasil analisis SHAP dalam Bahasa Indonesia yang
mudah dipahami oleh manajemen bank/fintech, regulator OJK, dan pelaku UMKM.
Gunakan analogi bisnis konteks Indonesia. Hindari jargon ML.
Format: paragraf naratif."""

top5 = shap_importance.head(5)
shap_ctx = "\n".join([
    f"- {r['feature']}: mean |SHAP| = {r['mean_abs_shap']:.4f}"
    for _, r in top5.iterrows()
])

USER_SHAP = f"""Hasil analisis SHAP untuk model credit risk scoring UMKM Indonesia:

5 Fitur Paling Berpengaruh:
{shap_ctx}

Framework: 4C (Capacity, Capital, Conditions, Character)
Dataset: 32.000+ pinjaman UMKM, default rate 22%

Keterangan fitur:
- debt_service_ratio = proporsi cicilan terhadap penghasilan
- loan_to_income_ratio = rasio pinjaman terhadap pendapatan
- income_log = log transformasi pendapatan
- interest_risk_band = band suku bunga (refleksi tingkat risiko)

Buat narasi 3-4 paragraf tentang:
1. Apa yang model pelajari tentang risiko kredit UMKM?
2. Mengapa Capacity mendominasi prediksi?
3. Implikasi praktis untuk kebijakan kredit UMKM Indonesia"""

print("⏳ Generating SHAP narrative...")
shap_narrative = call_genai(SYSTEM_SHAP, USER_SHAP, max_tokens=600)

print("\n" + "="*60)
print("NARASI SHAP — GPT-4o-mini")
print("="*60)
print(shap_narrative)

### 3B: Counterfactual Recommendation — Saran untuk Peminjam Ditolak

In [ ]:
# 3B: Counterfactual Recommendation
SYSTEM_CF = """Kamu adalah konsultan keuangan UMKM yang empatik dan praktis.
Ketika UMKM ditolak pinjaman, tugasmu:
1. Jelaskan alasan penolakan tanpa menyalahkan
2. Berikan 3-5 langkah konkret dalam 3-6 bulan ke depan
3. Estimasi perbaikan yang dibutuhkan
4. Akhiri dengan pesan motivasi
Tone: hangat, profesional, memberdayakan. Konteks: Indonesia, UMKM."""

hr_profile = X_sample.iloc[high_risk_idx]
hr_shap = sv[high_risk_idx]

shap_df_hr = pd.DataFrame({'feature': available_features, 'shap': hr_shap})
top_reasons = shap_df_hr[shap_df_hr['shap'] > 0].nlargest(3, 'shap')
reasons_text = "\n".join([
    f"- {r['feature']}: kontribusi +{r['shap']:.4f} ke risiko"
    for _, r in top_reasons.iterrows()
])

USER_CF = f"""Seorang pengusaha UMKM baru DITOLAK pinjaman.

Probabilitas default: {prob_sample[high_risk_idx]:.1%} (threshold: 50%)

Profil Peminjam:
- Debt Service Ratio: {hr_profile.get('debt_service_ratio', 'N/A'):.3f} (ideal: <0.35)
- Loan-to-Income Ratio: {hr_profile.get('loan_to_income_ratio', 'N/A'):.3f} (ideal: <0.3)
- Income Log: {hr_profile.get('income_log', 'N/A'):.3f}
- Interest Risk Band: {hr_profile.get('interest_risk_band', 'N/A'):.3f}

3 Faktor Penolakan Utama (SHAP analysis):
{reasons_text}

Apa yang harus dilakukan peminjam agar disetujui pada pengajuan berikutnya?"""

print("⏳ Generating counterfactual recommendations...")
counterfactual_advice = call_genai(SYSTEM_CF, USER_CF, max_tokens=700)

print("\n" + "="*60)
print("REKOMENDASI COUNTERFACTUAL — GPT-4o-mini")
print("="*60)
print(counterfactual_advice)

### 3C: Fairness Explanation — Analisis Bias & Inklusi Keuangan

In [ ]:
# 3C: Fairness Explanation
SYSTEM_FAIR = """Kamu adalah ekonom senior spesialis inklusi keuangan Indonesia.
Kamu memberikan analisis kebijakan untuk Bank Indonesia, OJK, dan Kemenkop UKM.
Analisismu harus berbasis data, relevan dengan konteks Indonesia, dan memberikan
rekomendasi actionable. Bedakan 'bias diskriminatif' vs 'refleksi realita ekonomi'."""

income_ctx = "\n".join([
    f"- {r['Kelompok']}: Default Rate={r['Actual Default Rate']}, Risk Score={r['Avg Risk Score']}"
    for _, r in fairness_income.iterrows()
])
age_ctx = "\n".join([
    f"- {r['Kelompok']}: Default Rate={r['Actual Default Rate']}, Risk Score={r['Avg Risk Score']}"
    for _, r in fairness_age.iterrows()
])

USER_FAIR = f"""Hasil fairness analysis model credit risk scoring UMKM:

Per Kelompok Pendapatan:
{income_ctx}
Income Disparity Gap (Q1 vs Q4): {gap_income:.1%}

Per Kelompok Usia:
{age_ctx}
Age Disparity Gap: {gap_age:.1%}

Konteks: 9 fitur 4C, 32.000+ data UMKM, tema Inklusi Keuangan Digital Indonesia.

Pertanyaan:
1. Apakah income disparity = 'bias diskriminatif' atau 'refleksi realita'? Jelaskan!
2. Apakah temuan 'age-blind' positif untuk inklusi keuangan?
3. Berikan 3 rekomendasi kebijakan konkret untuk mengatasi income gap
   tanpa mengorbankan akurasi model."""

print("⏳ Generating fairness explanation...")
fairness_explanation = call_genai(SYSTEM_FAIR, USER_FAIR, max_tokens=800)

print("\n" + "="*60)
print("ANALISIS FAIRNESS & INKLUSI KEUANGAN — GPT-4o-mini")
print("="*60)
print(fairness_explanation)

### 3D: Smart Policy Advisor — Chatbot Interaktif Multi-Turn

In [ ]:
# 3D: Smart Policy Advisor
SYSTEM_ADVISOR = f"""Kamu adalah UMKM Credit AI Advisor — asisten strategis kredit UMKM Indonesia.

DATA MODEL:
- Model: StackEnsemble AutoML (Azure ML)
- AUC: 0.9507 | Accuracy: 93.6% | F1: 93.3%
- Dataset: 32.000+ pinjaman UMKM, default rate 22%

SHAP INSIGHTS:
- Top 3: debt_service_ratio, loan_to_income_ratio, income_log
- Framework: 4C (Capacity, Capital, Conditions, Character)

FAIRNESS DATA:
- Income Gap (Q1 vs Q4): {gap_income:.1%}
- Age Gap: {gap_age:.1%} — model relatif age-blind (positif)

KONTEKS INDONESIA:
- 64 juta UMKM, 60%+ belum bankable
- Target inklusi keuangan OJK: 90% pada 2024

Jawab berbasis data di atas, dalam Bahasa Indonesia profesional, dengan rekomendasi actionable."""

def policy_advisor_chat(history: list, user_input: str) -> tuple:
    """Multi-turn chatbot dengan conversation history."""
    history.append({"role": "user", "content": user_input})
    try:
        response = client.chat.completions.create(
            model=ACTIVE_MODEL,
            messages=[{"role": "system", "content": SYSTEM_ADVISOR}] + history,
            max_tokens=600,
            temperature=0.7
        )
        reply = response.choices[0].message.content.strip()
        history.append({"role": "assistant", "content": reply})
        return reply, history
    except Exception as e:
        return f"[Error: {e}]", history

# Demo: 3 pertanyaan strategis
demo_questions = [
    "Bagaimana model ini bisa membantu inklusi keuangan UMKM di Indonesia?",
    "Apa risiko terbesar jika kita deploy model ini tanpa mempertimbangkan fairness?",
    "Berikan 3 rekomendasi konkret untuk bank/fintech yang ingin adopsi AI credit scoring ini."
]

conversation = []
print("\n" + "="*60)
print("SMART POLICY ADVISOR — DEMO CONVERSATION")
print("="*60)

for i, q in enumerate(demo_questions, 1):
    print(f"\n[{i}] User: {q}")
    print("⏳ ...")
    reply, conversation = policy_advisor_chat(conversation, q)
    print(f"\n[{i}] Advisor:\n{reply}")
    print("-"*40)

In [ ]:
# INTERACTIVE MODE — uncomment untuk demo presentasi langsung

# print("\nINTERACTIVE MODE — Smart Policy Advisor")
# print("Ketik 'exit' untuk keluar\n")
# 
# interactive_history = []
# while True:
#     user_input = input("\nKamu: ").strip()
#     if user_input.lower() in ['exit', 'quit', 'selesai']:
#         print("Sesi selesai.")
#         break
#     if not user_input:
#         continue
#     reply, interactive_history = policy_advisor_chat(interactive_history, user_input)
#     print(f"\nAdvisor: {reply}")

print("Tip: Uncomment block di atas untuk interactive demo saat presentasi!")

---
## Section 4: Export & Final Summary

In [ ]:
# Export semua hasil
import json
from datetime import datetime

genai_outputs = {
    "timestamp": datetime.now().isoformat(),
    "model_used": ACTIVE_MODEL,
    "endpoint": "https://models.inference.ai.azure.com",
    "shap_narrative": shap_narrative,
    "counterfactual_recommendation": counterfactual_advice,
    "fairness_explanation": fairness_explanation,
    "policy_advisor_demo": [
        {"turn": i+1, "role": m["role"], "content": m["content"]}
        for i, m in enumerate(conversation)
    ]
}

with open('genai_outputs.json', 'w', encoding='utf-8') as f:
    json.dump(genai_outputs, f, ensure_ascii=False, indent=2)

fairness_income.to_csv('fairness_income_metrics.csv', index=False)
fairness_age.to_csv('fairness_age_metrics.csv', index=False)
shap_importance.to_csv('shap_feature_importance.csv', index=False)

print("💾 Export selesai:")
for f in ['genai_outputs.json', 'fairness_income_metrics.csv',
          'fairness_age_metrics.csv', 'shap_feature_importance.csv',
          'shap_summary_plot.png', 'shap_waterfall_high_risk.png',
          'shap_waterfall_low_risk.png', 'fairness_analysis.png']:
    print(f"   ✅ {f}")

In [ ]:
# Final Summary
print("\n" + "="*65)
print("NOTEBOOK 03 COMPLETE — SUMMARY UNTUK PRESENTASI")
print("="*65)
print(f"""
SECTION 1 — SHAP ANALYSIS:
  Fitur terpenting: debt_service_ratio (Capacity mendominasi)
  Plots: Summary + 2 Waterfall (high/low risk)
  Framework 4C terbukti valid secara empiris

SECTION 2 — FAIRNESS ANALYSIS:
  Income Gap (Q1 vs Q4): {gap_income:.1%} — refleksi realita ekonomi
  Age Gap: {gap_age:.1%} — model Age-Blind (POSITIF untuk inklusi)
  Temuan mendukung tema inklusi keuangan

SECTION 3 — GENAI (GitHub Models gpt-4o-mini):
  3A: Narasi SHAP Bahasa Indonesia (untuk manajemen & regulator)
  3B: Counterfactual recommendation (untuk peminjam ditolak)
  3C: Fairness explanation (untuk kebijakan OJK/BI)
  3D: Smart Policy Advisor chatbot (multi-turn, context-aware)

DEPLOYMENT PATH (slide presentasi):
  GitHub Models (dev/demo) --> Azure OpenAI (production enterprise)
  OpenAI-compatible API = zero code change untuk migration

OUTPUT: 4 PNG | 3 CSV | 1 JSON
""")
print("NEXT: Streamlit demo app + Power BI dashboard!")
print("="*65)